In [160]:
from collections import Counter
import pandas as pd
import numpy as np
import joblib
import os

In [7]:
data_dir = "../output/calbicans_organism"
pathogen_code = "calbicans_organism"

# if args.organism == True:
#     tasks_dir = os.path.join(data_dir, "013a_raw_tasks_MOD")
# elif args.protein == True:
tasks_dir = os.path.join(data_dir, "013a_raw_tasks_MOD")


tasks = pd.read_csv(os.path.join(data_dir, "017_selected_tasks_RED.csv"))
tasks = tasks[tasks['RED'] == "1"]['task'].tolist()

In [151]:
def get_AGG_SCORE(tasks_dir, tasks):

    INACTIVES, AGG_SCORE = set(), {}

    for task in tasks:

        # Load data
        data = pd.read_csv(os.path.join(tasks_dir, task + ".csv"))
        activity = data.columns[-1]

        # Store actives in dict
        actives = sorted((set(data[data[activity] == 1]['inchikey'])))
        for act in actives:
            if act not in AGG_SCORE:
                AGG_SCORE[act] = []
            AGG_SCORE[act].append(task)

        # Store inactives in a set
        inactives = set(data[data[activity] == 0]['inchikey'])
        INACTIVES = INACTIVES.union(inactives)

    # Add inactives in AGG_SCORE - only if not already there
    for inact in INACTIVES:
        if inact not in AGG_SCORE:
            AGG_SCORE[inact] = []

    return AGG_SCORE

def calculate_reference_set(df, N=10000):
    df = df.copy()[['inchikey', 'num_tasks']]
    if len(df) < N:
        return df
    else:
        positives = df[df['num_tasks'] != 0][: int(N/2)]
        negatives = df[df['num_tasks'] == 0]
        if len(negatives) > int(N/2):
            negatives = negatives.sample(n=int(N/2), random_state=42)
        return pd.concat([positives, negatives])
    
def load_fingerprints(data_dir):
    X = np.load(os.path.join(data_dir, "014_fingerprints.npy"))
    with open(os.path.join(data_dir, "014_fingerprints_inchikeys.txt"), "r") as f:
        keys = f.read().splitlines()
    return X, keys

In [152]:
print("Calculating AGG Score")

# Get AGG Scores
AGG_SCORE = get_AGG_SCORE(tasks_dir, tasks)

# Sort by AGG Score
compounds_sorted = sorted(AGG_SCORE, key=lambda x: len(AGG_SCORE[x]), reverse=True)

# Store in a df
df = []
for cpd in compounds_sorted:
    df.append([cpd, len(AGG_SCORE[cpd]), ";".join(AGG_SCORE[cpd])]) 

df = pd.DataFrame(df, columns=['inchikey', 'num_tasks', 'tasks'])
df.to_csv(os.path.join(data_dir, '018_AGG_scores.csv'), index=False)

Calculating AGG Score


In [153]:
print("Generating reference set")

# Generate reference set
ref_set = calculate_reference_set(df, 10000)

# Storing reference set
ref_set.to_csv(os.path.join(data_dir, '018_ref_set.csv'), index=False)  # Maybe not needed

# Getting reference IKs
ref_IK = ref_set["inchikey"].tolist()

Generating reference set


In [154]:
# Loading fingerprints
X, inchikeys = load_fingerprints(data_dir)

# Preparing submatrix
indices = {}
for i, ik in enumerate(inchikeys):
    indices[ik] = i
idxs = [indices[ik] for ik in ref_IK]
X = X[idxs]

In [159]:
# Check that models are overfitted
...

In [ ]:
PREDICTIONS = {}

# For each task
for task in tasks:

    # Load overfitted models
    model = joblib.load(os.path.join(data_dir, ""))
    
    break

1_assay_CHEMBL4296189_MIC_pchembl_value_5_ORGANISM


In [161]:
selected_tasks = pd.read_csv(os.path.join(data_dir, "017_selected_tasks_RED.csv"))

In [162]:
selected_tasks

,task,auroc_avg_MOD,auroc_std_MOD,num_samples_MOD,num_pos_samples,pos:neg_MOD,auroc_avg_DIS,auroc_std_DIS,num_samples_DIS,pos:neg_DIS,SELECTED,RED,priority
0,1_assay_CHEMBL4296189_MIC_pchembl_value_5_ORGA...,0.8045,0.0672,624,77,0.1408,0.7926,0.0786,385,0.25,1,1,1
1,1_assay_CHEMBL4296189_MIC_pchembl_value_6_ORGA...,0.7602,0.1971,838,32,0.0397,0.7215,0.1524,160,0.25,1,1,1
2,1_assay_CHEMBL4296189_MIC_pchembl_percentile_1...,0.7493,0.0973,827,82,0.1101,0.8268,0.0487,410,0.25,1,1,1
3,1_assay_CHEMBL4296189_Inhibition_percentage_ac...,0.7049,0.0334,5234,265,0.0533,0.9060,0.0200,1325,0.25,1,1,1
4,1_assay_CHEMBL4296189_Inhibition_percentage_ac...,0.6957,0.0210,5234,531,0.1129,0.8943,0.0129,2655,0.25,3,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,5_grouped_percentiles_50_ORGANISM,0.9564,0.0082,28094,27605,56.4519,0.9413,0.0013,138025,0.25,2,1,5
87,5_grouped_percentiles_5_ORGANISM,0.8509,0.0051,27948,3337,0.1356,0.9392,0.0045,16685,0.25,1,1,5
88,5_grouped_percentiles_10_ORGANISM,0.8455,0.0056,28042,6532,0.3037,0.9473,0.0049,32660,0.25,1,1,5
89,5_grouped_percentiles_1_ORGANISM,0.8417,0.0252,27597,700,0.0260,0.9170,0.0245,3500,0.25,1,1,5
